In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.transaction_ml_pipeline import (
    CLUSTERED_DATASET_PATH,
    REPORTS_DIR,
    SUBMISSION_CLASSIFICATION_MODEL,
    SUBMISSION_CLUSTER_DATA,
)

In [2]:
training_df = pd.read_csv(SUBMISSION_CLUSTER_DATA if SUBMISSION_CLUSTER_DATA.exists() else CLUSTERED_DATASET_PATH)
training_df.head()

,TransactionAmount,TransactionType,MerchantCategory,Location,DeviceType,SpendingDeviationScore,VelocityScore,GeoAnomalyScore,PaymentChannel,Target
0,-0.027804,1.351387,1.521203,1.087041,-0.450874,-0.210076,-1.298980,-0.982700,0.433838,1
1,0.134752,1.351387,-0.661047,1.524341,-1.343929,-0.140220,-0.604988,1.597279,-1.353835,1
2,5.178774,-1.339487,-0.224597,-0.662160,0.442182,-1.776850,1.650488,1.353227,0.433838,0
3,2.805595,-1.339487,-0.661047,0.649741,0.442182,-0.599275,-0.778486,-0.459731,1.327675,0
4,-0.712028,0.454429,1.521203,1.524341,-0.450874,0.787869,0.436001,-0.808377,-1.353835,1


In [3]:
X = training_df.drop(columns=["Target"])
y = training_df["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
decision_tree = DecisionTreeClassifier(max_depth=6, random_state=42)
decision_tree.fit(X_train, y_train)
predictions = decision_tree.predict(X_test)
accuracy_score(y_test, predictions)

0.9993333333333333

In [5]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1407
           1       1.00      1.00      1.00      4593

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



In [6]:
joblib.dump(decision_tree, SUBMISSION_CLASSIFICATION_MODEL)
joblib.dump(decision_tree, PROJECT_ROOT / "models" / "decision_tree_model.h5")

['D:\\code\\Machine Learning\\models\\decision_tree_model.h5']